### Importing the packages

In [1]:
pip list

Package                                  Version
---------------------------------------- -----------
aiohappyeyeballs                         2.7.1
aiohttp                                  3.14.3
aiosignal                                1.4.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
anthropic                                0.115.1
anyio                                    4.14.1
astroid                                  4.0.4
asttokens                                3.0.1
attrs                                    26.1.0
bcrypt                                   5.0.0
build                                    1.5.0
certifi                                  2026.6.17
cffi                                     2.0.0
charset-normalizer                       3.4.7
chromadb                                 1.5.9
click                                    8.4.2
colorama                                 0.4.6
comm                                     0.

In [2]:
pip install chromadb==1.5.9

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install groq

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Load the Packages

In [4]:
import os
import json
from groq import Groq
from dotenv import load_dotenv
import chromadb


In [5]:
# Load environment variables from .env file
load_dotenv()

True

In [6]:
# Get API Key.
groq_api_key=os.getenv("GROQ_API_KEY")
print(groq_api_key[:10])

gsk_sHbDRt


In [7]:
# Creating Groq Client
client = Groq(api_key=groq_api_key)
print("Groq AI Client Created Successfully")

Groq AI Client Created Successfully


#### Communicating with GroqAI

In [8]:
response  = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages = [
        {"role": "system", "content": "You are a Helpful AI Assistant"},
        {"role": "user", "content": "Explain RAG in 3 bullet points"},
    ]
)

# print(response)
print(response.choices[0].message.content)

- **Combine retrieval with generation:** RAG first fetches relevant documents (or chunks) from an external knowledge base using a dense retriever, then feeds those retrieved texts to a generative language model, which conditions its response on both the user query and the retrieved evidence.  

- **Improves accuracy & factuality:** Because the generator can cite up‑to‑date or domain‑specific information rather than relying solely on its internal parameters, RAG reduces hallucinations and keeps answers aligned with the latest or specialized data.  

- **Modular, scalable architecture:** The retrieval component (e.g., BM25, DPR, or vector‑search) can be swapped or scaled independently of the generator (e.g., GPT, T5), allowing easy updates to the knowledge source without retraining the entire model.


In [9]:
# JSON Representation

response.model_dump_json()

# If we want to convert RAG to Agentic RAG, finish_reason: tool_call

'{"id":"chatcmpl-eb948a3a-ffc7-4ebe-a854-0695386e535d","choices":[{"finish_reason":"stop","index":0,"logprobs":null,"message":{"content":"- **Combine retrieval with generation:** RAG first fetches relevant documents (or chunks) from an external knowledge base using a dense retriever, then feeds those retrieved texts to a generative language model, which conditions its response on both the user query and the retrieved evidence.  \\n\\n- **Improves accuracy & factuality:** Because the generator can cite up‑to‑date or domain‑specific information rather than relying solely on its internal parameters, RAG reduces hallucinations and keeps answers aligned with the latest or specialized data.  \\n\\n- **Modular, scalable architecture:** The retrieval component (e.g., BM25, DPR, or vector‑search) can be swapped or scaled independently of the generator (e.g., GPT, T5), allowing easy updates to the knowledge source without retraining the entire model.","role":"assistant","annotations":null,"execu

#### STEP 2 - LOADING THE DATA AND CHUNKING THE DATA FROM DOCUMENTS (_data folder)

In [10]:
with open("F:\\Career\\ai_engineer\\RAG_Project\\_data\\company_hr_policy.txt", "r") as file:
    hr_document = file.read()

print(hr_document)

COMPANY HUMAN RESOURCES POLICY MANUAL - TEMPLATE

1. PURPOSE AND SCOPE
- Purpose: To define standard expectations, rules, and benefits for all team members.
- Scope: Applies to all full-time and part-time employees of [Company Name].

2. WORKING HOURS AND ATTENDANCE
- Work Week: Monday through Friday, 9:00 AM to 6:00 PM.
- Break Time: One-hour lunch break daily.
- Punctuality: Employees must log attendance at the start and end of shifts.
- Overtime: Must be approved by a manager in advance.

3. LEAVE AND TIME OFF
- Casual Leave (CL): 12 days per year, accrued monthly.
- Sick Leave (SL): 12 days per year for personal illness.
- Public Holidays: List of official company holidays published annually.

4. CODE OF CONDUCT
- Professionalism: Maintain respectful communication with peers and clients.
- Dress Business casual or formal as per department norms.
- Prohibitions: Zero tolerance for harassment, discrimination, or substance abuse on premises.

5. PERFORMANCE AND SEPARATION
- Reviews: A

In [11]:
with open("..\\RAG_Project\\_data\\onboarding_guide.txt", "r") as file:
    onboarding_document = file.read()

print(onboarding_document)

EMPLOYEE ONBOARDING GUIDE & CHECKLIST

Employee Name: [Insert Name]
Position Title: [Insert Title]
Department: [Insert Department]
Start Date: [MM/DD/YYYY]
Manager/Mentor: [Insert Name]

--------------------------------------------------
1. PRE-ARRIVAL (Before Day 1)
--------------------------------------------------
[ ] Send official welcome email with start details
[ ] Setup computer, email address, and system logins
[ ] Prepare desk, chair, and access badge/ID
[ ] Assign an onboarding buddy or mentor

--------------------------------------------------
2. DAY ONE: ORIENTATION
--------------------------------------------------
[ ] 10:00 AM - Welcome session and team introduction
[ ] 11:00 AM - Complete HR paperwork and tax forms
[ ] 12:00 PM - IT setup check and system access walkthrough
[ ] 02:00 PM - Department overview with direct manager
[ ] 03:00 PM - Office tour and facility orientation

--------------------------------------------------
3. WEEK ONE: INTEGRATION
----------------

In [12]:
# Length of documents

print(f"HR Policy Document - Characters Length: ", len(hr_document))
print(f"Onboarding Guide Document - Characters Length: ", len(onboarding_document))

# No of words in documents.
print(f"HR Policy Document - Words: ", len(hr_document.split()))
print(f"Onboarding Guide Document - Words: ", len(onboarding_document.split()))



HR Policy Document - Characters Length:  1236
Onboarding Guide Document - Characters Length:  1712
HR Policy Document - Words:  181
Onboarding Guide Document - Words:  200


#### Next Chunking strategy on the documents using Python

In [13]:
result = hr_document.strip().split("\n")
print(result)


['==================================================', 'COMPANY HUMAN RESOURCES POLICY MANUAL - TEMPLATE', '==================================================', '', '1. PURPOSE AND SCOPE', '- Purpose: To define standard expectations, rules, and benefits for all team members.', '- Scope: Applies to all full-time and part-time employees of [Company Name].', '', '2. WORKING HOURS AND ATTENDANCE', '- Work Week: Monday through Friday, 9:00 AM to 6:00 PM.', '- Break Time: One-hour lunch break daily.', '- Punctuality: Employees must log attendance at the start and end of shifts.', '- Overtime: Must be approved by a manager in advance.', '', '3. LEAVE AND TIME OFF', '- Casual Leave (CL): 12 days per year, accrued monthly.', '- Sick Leave (SL): 12 days per year for personal illness.', '- Public Holidays: List of official company holidays published annually.', '', '4. CODE OF CONDUCT', '- Professionalism: Maintain respectful communication with peers and clients.', '- Dress Business casual or for

In [17]:
def chunk_documents(text, tag_name):
    # Split on double newlines to separate paragraphs
    paragraph = text.strip().split("\n")

    chunks = []

    for para in paragraph:
        para = para.strip()
        if len(para) < 50:
            continue
        if para.startswith(("==","---")):
            continue
        
        chunks.append({"text": para, "source": tag_name})

    # Returned AFTER processing all paragraphs
    return chunks

    
hr_chunks = chunk_documents(hr_document, "Company_hr_policy")
print(f"Total chunks created: {len(hr_chunks)}")
print(hr_chunks)

print("\n")

onboarding_chunks = chunk_documents(onboarding_document,"Onboarding_Guide")
print(f"Total chunks created: {len(onboarding_chunks)}")
print(onboarding_chunks)


Total chunks created: 13
[{'text': '- Purpose: To define standard expectations, rules, and benefits for all team members.', 'source': 'Company_hr_policy'}, {'text': '- Scope: Applies to all full-time and part-time employees of [Company Name].', 'source': 'Company_hr_policy'}, {'text': '- Work Week: Monday through Friday, 9:00 AM to 6:00 PM.', 'source': 'Company_hr_policy'}, {'text': '- Punctuality: Employees must log attendance at the start and end of shifts.', 'source': 'Company_hr_policy'}, {'text': '- Overtime: Must be approved by a manager in advance.', 'source': 'Company_hr_policy'}, {'text': '- Casual Leave (CL): 12 days per year, accrued monthly.', 'source': 'Company_hr_policy'}, {'text': '- Sick Leave (SL): 12 days per year for personal illness.', 'source': 'Company_hr_policy'}, {'text': '- Public Holidays: List of official company holidays published annually.', 'source': 'Company_hr_policy'}, {'text': '- Professionalism: Maintain respectful communication with peers and clients